# Module 1, Notebook 2: Policy Iteration and Value Iteration

## Learning Objectives
By completing this notebook, you will:
1. Implement policy iteration (full evaluation + greedy improvement) from scratch
2. Implement value iteration (one Bellman optimality backup per sweep) from scratch
3. Compare convergence speed and policy quality on a FrozenLake-like GridWorld
4. Visualize optimal policies as arrow grids
5. Understand why value iteration is strictly more efficient than policy iteration for large MDPs

## Prerequisites
- Module 1, Notebook 1: policy evaluation, iterative Bellman updates, convergence tracking
- Module 0, Notebook 3: Bellman optimality equations

## Estimated Time: 15 minutes

---

**Key Insight:** Both policy iteration and value iteration converge to the same optimal policy $\pi^*$, but they get there differently. Policy iteration takes big steps (full evaluation) but makes guaranteed progress at each step. Value iteration takes tiny steps (one Bellman backup) but runs much faster per iteration.

## 1. Setup

In [ ]:
# Purpose: Imports and reproducibility
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm

np.random.seed(42)
print("Ready. NumPy version:", np.__version__)

## 2. FrozenLake-Style GridWorld

We use a 4x4 FrozenLake-style environment with frozen cells (safe), holes (penalty, terminal), and a goal (reward, terminal). Unlike the S&B GridWorld from Notebook 1, here we distinguish between:
- **Holes**: reward -1, episode ends
- **Goal**: reward +1, episode ends  
- **Frozen**: reward 0, episode continues

This richer reward structure makes the optimal policy more interesting to visualize.

In [ ]:
# Purpose: Build a FrozenLake-style GridWorld MDP
# Key Concept: Positive and negative terminal rewards create a non-trivial optimal policy

# Grid: S=start, F=frozen, H=hole, G=goal
# Matches standard 4x4 FrozenLake layout
FROZEN_LAKE_MAP = [
    'SFFF',
    'FHFH',
    'FFFH',
    'HFFG'
]

N_ROWS, N_COLS = 4, 4
N_STATES  = N_ROWS * N_COLS
N_ACTIONS = 4

# Parse map
CELL_TYPES = {}  # state index -> cell character
for r, row in enumerate(FROZEN_LAKE_MAP):
    for c, ch in enumerate(row):
        CELL_TYPES[r * N_COLS + c] = ch

TERMINAL_STATES = {s for s, ch in CELL_TYPES.items() if ch in ('H', 'G')}

# Rewards per cell type entered
CELL_REWARDS = {'S': 0.0, 'F': 0.0, 'H': -1.0, 'G': +1.0}

DELTAS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
ACTION_ARROWS = {0: '↑', 1: '↓', 2: '←', 3: '→'}

def pos_to_state(r, c):
    return r * N_COLS + c

def state_to_pos(s):
    return s // N_COLS, s % N_COLS


def build_frozenlake_mdp():
    """
    Build deterministic FrozenLake MDP.

    Returns
    -------
    P : np.ndarray (n_states, n_actions, n_states)
    R : np.ndarray (n_states, n_actions)
    """
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))

    for s in range(N_STATES):
        if s in TERMINAL_STATES:
            for a in range(N_ACTIONS):
                P[s, a, s] = 1.0
                R[s, a]    = 0.0
            continue

        r, c = state_to_pos(s)
        for a in range(N_ACTIONS):
            dr, dc = DELTAS[a]
            nr, nc = r + dr, c + dc

            # Wall collision
            if not (0 <= nr < N_ROWS and 0 <= nc < N_COLS):
                nr, nc = r, c

            s_next = pos_to_state(nr, nc)
            P[s, a, s_next] += 1.0
            R[s, a] = CELL_REWARDS[CELL_TYPES[s_next]]

    assert np.allclose(P.sum(axis=2), 1.0)
    return P, R


P_fl, R_fl = build_frozenlake_mdp()

print("FrozenLake map:")
for r, row in enumerate(FROZEN_LAKE_MAP):
    print('  ' + '  '.join(row))
print(f"\nTerminal states: {sorted(TERMINAL_STATES)}")
print(f"  Holes (H, reward=-1): {sorted(s for s in TERMINAL_STATES if CELL_TYPES[s]=='H')}")
print(f"  Goal  (G, reward=+1): {sorted(s for s in TERMINAL_STATES if CELL_TYPES[s]=='G')}")

## 3. Policy Iteration

Policy iteration alternates two steps until the policy stops changing:

1. **Policy Evaluation**: compute $V^{\pi_k}$ exactly (or to convergence)
2. **Policy Improvement**: set $\pi_{k+1}(s) = \arg\max_a Q^{\pi_k}(s,a)$

**Guarantee**: Each improvement step produces a strictly better policy (or the current policy is already optimal). The process must terminate in a finite number of steps because there are finitely many deterministic policies.

In [ ]:
# Purpose: Implement the policy evaluation subroutine used inside policy iteration
# Key Concept: Policy iteration requires running evaluation to near-convergence at each step

def evaluate_policy_fast(P, R, policy, gamma=0.99, theta=1e-8):
    """
    Iterative policy evaluation — returns final V only.

    Parameters
    ----------
    P : np.ndarray (n_states, n_actions, n_states)
    R : np.ndarray (n_states, n_actions)
    policy : np.ndarray (n_states, n_actions)  — stochastic policy
    gamma : float
    theta : float

    Returns
    -------
    V : np.ndarray (n_states,)
    n_sweeps : int
    """
    n_states = P.shape[0]
    V = np.zeros(n_states)
    n_sweeps = 0

    while True:
        V_new = np.zeros(n_states)
        for s in range(n_states):
            # Vectorized Bellman update
            expected_next = P[s].dot(V)           # (n_actions,)
            action_values = R[s] + gamma * expected_next
            V_new[s] = policy[s].dot(action_values)
        n_sweeps += 1
        if np.max(np.abs(V_new - V)) < theta:
            V = V_new
            break
        V = V_new

    return V, n_sweeps


def improve_policy(P, R, V, gamma=0.99):
    """
    Greedy policy improvement step.

    Given V^pi, compute a new deterministic policy that is greedy w.r.t. Q^pi.

    Returns
    -------
    new_policy : np.ndarray (n_states, n_actions)  — deterministic (one-hot)
    policy_stable : bool
        True if policy did not change (algorithm has converged).
    old_actions : np.ndarray (n_states,)
    new_actions : np.ndarray (n_states,)
    """
    n_states, n_actions = P.shape[0], P.shape[1]

    # Compute Q(s,a) = R[s,a] + gamma * sum_s' P[s,a,s'] * V[s']
    expected_next_V = np.einsum('sas,s->sa', P, V)  # (n_states, n_actions)
    Q = R + gamma * expected_next_V

    old_actions = np.argmax(
        np.einsum('sa,sa->sa', P.sum(axis=2), np.ones((n_states, n_actions))), axis=1
    )  # placeholder; overwritten below

    new_actions = np.argmax(Q, axis=1)
    new_policy = np.zeros((n_states, n_actions))
    for s in range(n_states):
        new_policy[s, new_actions[s]] = 1.0

    return new_policy, Q, new_actions


print("Policy evaluation and improvement utilities defined.")

In [ ]:
# Purpose: Run the full policy iteration algorithm
# Key Concept: Policy iteration converges in very few outer iterations for small MDPs

def policy_iteration(P, R, gamma=0.99, eval_theta=1e-8, max_outer_iter=100):
    """
    Full policy iteration: evaluate -> improve -> repeat until stable.

    Returns
    -------
    V_star : np.ndarray (n_states,)  — optimal value function
    pi_star : np.ndarray (n_states, n_actions)  — optimal policy (one-hot)
    history : list of dict
        Per-iteration record with keys: 'iteration', 'n_eval_sweeps', 'V', 'actions'
    """
    n_states, n_actions = P.shape[0], P.shape[1]

    # Start with uniform random policy
    policy = np.ones((n_states, n_actions)) / n_actions
    V = np.zeros(n_states)
    history = []

    for iteration in range(max_outer_iter):
        # Step 1: Evaluate current policy
        V, n_sweeps = evaluate_policy_fast(P, R, policy, gamma, eval_theta)

        # Step 2: Improve policy greedily
        old_actions = np.argmax(policy, axis=1)
        new_policy, Q, new_actions = improve_policy(P, R, V, gamma)

        history.append({
            'iteration':     iteration + 1,
            'n_eval_sweeps': n_sweeps,
            'V':             V.copy(),
            'actions':       new_actions.copy()
        })

        # Check stability: did the policy change?
        policy_stable = np.all(old_actions == new_actions)
        policy = new_policy

        print(f"Iter {iteration+1:2d}: eval_sweeps={n_sweeps:4d}, "
              f"policy_stable={policy_stable}")

        if policy_stable:
            print(f"Policy iteration converged after {iteration+1} outer iterations.")
            break

    return V, policy, history


GAMMA = 0.99
print(f"Running policy iteration with gamma={GAMMA}...")
V_pi_star, pi_star_pi, pi_history = policy_iteration(P_fl, R_fl, gamma=GAMMA)

## 4. Value Iteration

Value iteration collapses the two steps into one. Instead of the Bellman expectation operator, it applies the **Bellman optimality operator**:

$$V_{k+1}(s) \leftarrow \max_a \sum_{s'} P(s'|s,a)\bigl[R(s,a,s') + \gamma V_k(s')\bigr]$$

The key difference: `sum_a pi(a|s) [...]` becomes `max_a [...]`. This is also a contraction with constant $\gamma$, so it converges to $V^*$. Then the optimal policy is extracted as a one-shot greedy step at the end.

In [ ]:
# Purpose: Implement value iteration
# Key Concept: Value iteration uses the Bellman optimality operator (max over actions)

def value_iteration(P, R, gamma=0.99, theta=1e-8, max_iter=1000):
    """
    Value iteration: repeatedly apply Bellman optimality backup until convergence.

    Parameters
    ----------
    P : np.ndarray (n_states, n_actions, n_states)
    R : np.ndarray (n_states, n_actions)
    gamma : float
    theta : float
    max_iter : int

    Returns
    -------
    V_star : np.ndarray (n_states,)
    pi_star : np.ndarray (n_states, n_actions)  — greedy policy from V_star
    delta_history : list of float
    V_history : list of np.ndarray  — snapshots every 10 iterations
    """
    n_states, n_actions, _ = P.shape
    V = np.zeros(n_states)
    delta_history = []
    V_history     = [V.copy()]

    for iteration in range(1, max_iter + 1):
        V_new = np.zeros(n_states)

        for s in range(n_states):
            # Bellman optimality backup: V_new[s] = max_a [R[s,a] + gamma * sum_s' P[s,a,s'] V[s']]
            expected_next_V = P[s].dot(V)           # (n_actions,)
            action_values   = R[s] + gamma * expected_next_V  # Q(s, a) for each a
            V_new[s]        = np.max(action_values)  # <-- MAX instead of weighted average

        max_delta = np.max(np.abs(V_new - V))
        delta_history.append(max_delta)
        V = V_new

        if iteration % 10 == 0:
            V_history.append(V.copy())

        if max_delta < theta:
            V_history.append(V.copy())
            print(f"Value iteration converged in {iteration} sweeps (max delta = {max_delta:.2e})")
            break

    # Extract optimal policy from converged V
    expected_next_V = np.einsum('sas,s->sa', P, V)  # (n_states, n_actions)
    Q_star  = R + gamma * expected_next_V
    actions = np.argmax(Q_star, axis=1)

    pi_star = np.zeros((n_states, n_actions))
    for s in range(n_states):
        pi_star[s, actions[s]] = 1.0

    return V, pi_star, delta_history, V_history


print(f"Running value iteration with gamma={GAMMA}...")
V_vi_star, pi_star_vi, vi_deltas, vi_V_history = value_iteration(P_fl, R_fl, gamma=GAMMA)

# Check that policy iteration and value iteration found the same optimal policy
pi_pi_actions = np.argmax(pi_star_pi, axis=1)
pi_vi_actions = np.argmax(pi_star_vi, axis=1)
policies_agree = np.all(pi_pi_actions == pi_vi_actions)
print(f"\nPolicies agree: {policies_agree}")
print(f"Max V* difference: {np.max(np.abs(V_pi_star - V_vi_star)):.2e}")

## 5. Visualizing Optimal Policies as Arrow Grids

An arrow grid maps each state to the greedy action as a directional arrow. Holes and the goal are marked with their cell labels. This is the standard way to display an RL policy for 2D environments.

In [ ]:
# Purpose: Visualize V* and pi* as arrow grids for both methods
# Key Concept: Arrow grids make the policy human-interpretable

def plot_policy_arrow_grid(V, policy_actions, cell_types, terminal_states,
                           n_rows, n_cols, ax, title):
    """
    Draw a value heatmap with policy arrows overlaid.

    Parameters
    ----------
    V : np.ndarray (n_states,)
    policy_actions : np.ndarray (n_states,)  — greedy action per state
    cell_types : dict  — state -> 'S','F','H','G'
    terminal_states : set
    n_rows, n_cols : int
    ax : matplotlib Axes
    title : str
    """
    V_grid = V.reshape(n_rows, n_cols)

    # Heatmap
    vmin = V_grid.min()
    vmax = V_grid.max()
    vcenter = 0.0 if vmin < 0 < vmax else (vmin + vmax) / 2
    norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=max(vmax, vcenter + 0.01))
    im = ax.imshow(V_grid, cmap='RdYlGn', norm=norm, interpolation='nearest')

    cell_labels = {'S': 'S', 'F': '', 'H': 'H', 'G': 'G'}
    cell_colors = {'S': 'white', 'F': 'black', 'H': 'white', 'G': 'white'}

    for r in range(n_rows):
        for c in range(n_cols):
            s = r * n_cols + c
            ct = cell_types[s]

            # Value label
            ax.text(c, r - 0.3, f'{V_grid[r, c]:.2f}', ha='center', va='center',
                    fontsize=7.5, color=cell_colors[ct])

            # Policy arrow or cell label for terminals
            if s in terminal_states:
                ax.text(c, r + 0.2, cell_labels[ct], ha='center', va='center',
                        fontsize=14, color=cell_colors[ct], fontweight='bold')
            else:
                arrow = ACTION_ARROWS[policy_actions[s]]
                ax.text(c, r + 0.2, arrow, ha='center', va='center',
                        fontsize=16, color='#1c7ed6', fontweight='bold')

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    return im


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = plot_policy_arrow_grid(
    V_pi_star, np.argmax(pi_star_pi, axis=1),
    CELL_TYPES, TERMINAL_STATES, N_ROWS, N_COLS,
    axes[0], 'Policy Iteration — $V^*$ and $\\pi^*$'
)

im2 = plot_policy_arrow_grid(
    V_vi_star, np.argmax(pi_star_vi, axis=1),
    CELL_TYPES, TERMINAL_STATES, N_ROWS, N_COLS,
    axes[1], 'Value Iteration — $V^*$ and $\\pi^*$'
)

plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label='V*(s)')
plt.tight_layout()
plt.savefig('optimal_policies.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to optimal_policies.png")

## 6. Convergence Comparison

Policy iteration requires very few outer iterations (policy improvement steps) but many inner sweeps (evaluation). Value iteration requires many sweeps overall but does not distinguish inner/outer loops.

We compare total sweep counts and convergence curves.

In [ ]:
# Purpose: Compare total computation (sweeps) between the two methods
# Key Concept: Value iteration's low overhead per iteration typically wins for large MDPs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Value Iteration convergence curve ---
ax = axes[0]
ax.semilogy(vi_deltas, color='#1c7ed6', linewidth=2, label='Value Iteration')
ax.set_xlabel('Sweep', fontsize=12)
ax.set_ylabel('Max $|\\Delta V|$ (log scale)', fontsize=12)
ax.set_title('Value Iteration — Convergence Curve', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
ax.axhline(1e-8, color='red', linestyle='--', linewidth=1, label='$\\theta=10^{-8}$')
ax.legend(fontsize=10)

# --- Right: Policy Iteration — sweeps per outer iteration ---
ax = axes[1]
pi_iters   = [h['iteration'] for h in pi_history]
pi_sweeps  = [h['n_eval_sweeps'] for h in pi_history]
cumulative = np.cumsum(pi_sweeps)

ax.bar(pi_iters, pi_sweeps, color='#e64980', alpha=0.8, label='Eval sweeps per iteration')
ax2 = ax.twinx()
ax2.plot(pi_iters, cumulative, 'o-', color='#f76707', linewidth=2,
         label='Cumulative sweeps')
ax2.set_ylabel('Cumulative sweeps', fontsize=11, color='#f76707')
ax2.tick_params(axis='y', labelcolor='#f76707')

ax.set_xlabel('Outer iteration', fontsize=12)
ax.set_ylabel('Evaluation sweeps', fontsize=12, color='#e64980')
ax.tick_params(axis='y', labelcolor='#e64980')
ax.set_title('Policy Iteration — Sweeps per Outer Iteration', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax2.legend(fontsize=10, loc='center right')
ax.set_xticks(pi_iters)

plt.tight_layout()
plt.savefig('convergence_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Policy iteration: {len(pi_history)} outer iterations, "
      f"{sum(pi_sweeps)} total evaluation sweeps")
print(f"Value iteration:  {len(vi_deltas)} sweeps total")

## 7. Exercises

### Exercise 2.1: Truncated Policy Evaluation

**Task:** In standard policy iteration, evaluation runs until convergence (many sweeps). **Truncated policy iteration** runs evaluation for exactly $k$ sweeps instead. Implement `policy_iteration_truncated(P, R, gamma, k_eval, max_outer)` that limits each evaluation phase to exactly `k_eval` sweeps. Run it with `k_eval=1` (this is exactly value iteration!) and `k_eval=5` on the FrozenLake MDP. Count the total number of Bellman backups for each and compare to the standard methods.

**Expected Output:** Total Bellman backups and final V[0] for each configuration.

**Hints:**
<details>
<summary>Hint 1</summary>
Modify `evaluate_policy_fast` to accept `max_sweeps` and replace the `while True` loop with a `for _ in range(max_sweeps)` loop.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
The outer loop can then run until the policy stops changing. Track total sweeps as `n_outer_iters * k_eval` — but be careful: the outer loop may also need a max iteration limit to avoid infinite loops.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
def policy_iteration_truncated(P, R, gamma=0.99, k_eval=1, max_outer=500):
    """
    Policy iteration with truncated evaluation (exactly k_eval sweeps per evaluation).

    Returns
    -------
    V : np.ndarray (n_states,)
    pi : np.ndarray (n_states, n_actions)
    total_backups : int  — total Bellman backup operations across all iterations
    n_outer : int        — number of outer iterations
    """
    n_states, n_actions = P.shape[0], P.shape[1]
    policy = np.ones((n_states, n_actions)) / n_actions  # start uniform
    V = np.zeros(n_states)
    total_backups = 0

    for outer_iter in range(max_outer):
        # Step 1: Truncated evaluation (exactly k_eval sweeps)
        # ... your code here ...
        pass

        # Step 2: Greedy improvement
        # ... your code here ...
        pass

        # Step 3: Check policy stability and break if stable
        # ... your code here ...
        break  # Remove this break once your loop is implemented

    return V, policy, total_backups, outer_iter + 1


# Test with k_eval=1 and k_eval=5
V_k1, pi_k1, backups_k1, n_outer_k1 = policy_iteration_truncated(P_fl, R_fl, gamma=GAMMA, k_eval=1)
V_k5, pi_k5, backups_k5, n_outer_k5 = policy_iteration_truncated(P_fl, R_fl, gamma=GAMMA, k_eval=5)

print(f"k_eval=1: {n_outer_k1} outer iters, {backups_k1} total backups, V[0]={V_k1[0]:.4f}")
print(f"k_eval=5: {n_outer_k5} outer iters, {backups_k5} total backups, V[0]={V_k5[0]:.4f}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_1():
    # Both truncated versions should find policies with V similar to the exact methods
    assert V_k1 is not None and V_k5 is not None, "Run policy_iteration_truncated for both k values!"
    assert V_k1.shape == (N_STATES,), f"V_k1 shape should be ({N_STATES},)"
    assert V_k5.shape == (N_STATES,), f"V_k5 shape should be ({N_STATES},)"

    # Values at the goal state should be positive
    goal_state = [s for s, ch in CELL_TYPES.items() if ch == 'G'][0]
    assert V_k1[goal_state] >= 0, f"V[goal] should be >= 0 with k_eval=1, got {V_k1[goal_state]:.3f}"
    assert V_k5[goal_state] >= 0, f"V[goal] should be >= 0 with k_eval=5, got {V_k5[goal_state]:.3f}"

    # k_eval=5 should require fewer outer iterations than k_eval=1
    assert n_outer_k5 <= n_outer_k1, \
        (f"k_eval=5 should need fewer outer iterations than k_eval=1. "
         f"Got k5={n_outer_k5}, k1={n_outer_k1}")

    # Total backups: k_eval=5 should have more per outer but fewer total (or comparable)
    assert backups_k1 > 0, "Total backups must be positive."
    assert backups_k5 > 0, "Total backups must be positive."

    print(f"Exercise 2.1 passed!")
    print(f"  k=1: {n_outer_k1} outer iters x 1 sweep = {backups_k1} total backups")
    print(f"  k=5: {n_outer_k5} outer iters x 5 sweeps = {backups_k5} total backups")

test_exercise_2_1()

### Exercise 2.2: Sensitivity to Gamma in Optimal Policy

**Task:** Run value iteration with $\gamma \in \{0.5, 0.8, 0.99\}$ on the FrozenLake MDP. For each gamma, extract the optimal policy as a flat array of actions. Count how many states have different optimal actions between $\gamma=0.5$ and $\gamma=0.99$. Visualize the optimal value functions for all three gammas as side-by-side heatmaps.

**Expected Output:** A printed count of differing states and a matplotlib figure.

**Hints:**
<details>
<summary>Hint 1</summary>
Use the existing `value_iteration` function three times with different `gamma` values and collect the results.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
gammas = [0.5, 0.8, 0.99]
vi_results = {}  # gamma -> {'V': ..., 'pi': ..., 'actions': ...}

for g in gammas:
    pass  # Replace with: run value_iteration and store V, pi, and argmax(pi)

# Count differing actions between gamma=0.5 and gamma=0.99
n_different_states = None  # Replace with integer

# Plot heatmaps (optional for extra credit — the count is required)
print(f"States with different optimal action (gamma=0.5 vs 0.99): {n_different_states}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_2():
    assert len(vi_results) == 3, f"Run value iteration for all 3 gammas. Got {len(vi_results)} results."

    for g in gammas:
        assert g in vi_results, f"Missing result for gamma={g}"
        assert 'V' in vi_results[g], f"Missing V for gamma={g}"
        assert 'actions' in vi_results[g], f"Missing actions for gamma={g}"
        assert vi_results[g]['V'].shape == (N_STATES,), \
            f"V for gamma={g} has wrong shape."

    # Higher gamma should generally produce higher V at non-terminal states
    v_goal = [s for s, ch in CELL_TYPES.items() if ch == 'G'][0]
    # V at goal state is 0 for all (absorbing), so compare a reachable state
    reachable = [s for s in range(N_STATES) if s not in TERMINAL_STATES]
    if reachable:
        s_test = reachable[0]
        v_low  = vi_results[0.5]['V'][s_test]
        v_high = vi_results[0.99]['V'][s_test]
        assert v_high >= v_low - 0.1, \
            f"V[{s_test}] should be >= with higher gamma. Got {v_low:.3f} vs {v_high:.3f}"

    assert n_different_states is not None, "Compute n_different_states!"
    assert isinstance(n_different_states, (int, np.integer)), \
        "n_different_states should be an integer."
    assert 0 <= n_different_states <= N_STATES, \
        f"n_different_states must be in [0, {N_STATES}], got {n_different_states}"

    print(f"Exercise 2.2 passed!")
    print(f"  States with different optimal actions (gamma=0.5 vs 0.99): {n_different_states}")
    print(f"  V[0] by gamma: " + ", ".join(f"{g}: {vi_results[g]['V'][0]:.3f}" for g in gammas))

test_exercise_2_2()

## Key Takeaways

1. **Policy iteration** = evaluate fully + improve greedily, repeat. Converges in few outer iterations but each evaluation phase is expensive.
2. **Value iteration** = apply the Bellman optimality operator (max instead of expectation) until convergence. One combined loop, no inner/outer distinction.
3. **Both methods find the same optimal policy** $\pi^*$ — they differ only in computational path, not destination.
4. **Truncated policy evaluation** with $k=1$ sweep per improvement step is exactly value iteration. As $k \to \infty$, it approaches standard policy iteration.
5. **Visualizing policies as arrow grids** is the gold standard for 2D environments — it immediately reveals whether the agent is navigating toward the goal or into holes.
6. **The discount factor shapes the optimal policy**: with very low $\gamma$, the agent may choose suboptimal long-term paths because it underweights distant rewards.

## What's Next

Module 2 (`01_mc_prediction.ipynb`) introduces **Monte Carlo methods** — the first class of algorithms that do not require a model of the environment ($P$ and $R$). Instead of computing Bellman backups exactly, Monte Carlo methods sample complete trajectories and estimate values from the returns. This is the bridge from dynamic programming to model-free RL.